# Notebook 01 — Data preparation

**Paper:** *Stochastic collapse and first-passage dynamics in marathon pacing: a Kramers–Moyal approach*  
**Author:** A. Domínguez-Monterroza  
**Target:** Physical Review E

This notebook preprocesses official split-time records from four World Marathon Majors
and produces the two CSV files consumed by Notebook 02:

| Output file | Contents |
|-------------|----------|
| `segment_paces_data.csv` | Segment-wise pace values and state variables per runner |
| `wall_labels_data.csv` | Collapse labels and first-passage distances |

---

## Data sources

| Marathon | Editions | Official results URL |
|----------|----------|---------------------|
| Boston | 2021–2024 | https://www.baa.org/races/boston-marathon/results |
| Chicago | 2021–2024 | https://www.chicagomarathon.com/runners/results/ |
| New York City | 2021–2024 | https://results.nyrr.org/ |
| Berlin | 2021–2024 | https://www.bmw-berlin-marathon.com/en/facts-and-figures/results-archive/ |

Download the raw results files (CSV or Excel) from the URLs above and place them
in the `data/raw/` directory before running this notebook.

## 1. Imports and configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd

print('Imports OK')

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
RAW_DIR     = Path('data/raw')        # directory with downloaded race results
OUTPUT_DIR  = Path('data/processed')  # output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Cleaning thresholds (Paper §II) ───────────────────────────────────────────
PACE_MIN_KM  = 2.0    # min/km — physiological lower bound
PACE_MAX_KM  = 20.0   # min/km — physiological upper bound
FINISH_MIN   = 120.0  # minutes
FINISH_MAX   = 480.0  # minutes

# ── Collapse definition (Paper §II.C) ─────────────────────────────────────────
Y_THRESHOLD    = 0.10  # primary state threshold y_c
LATE_START_IDX = 5     # segment index corresponding to ≥ 25 km

# ── Segment structure ─────────────────────────────────────────────────────────
SEG_ORDER = [
    '0_5k', '5_10k', '10_15k', '15_20k', '20k_half',
    'half_25k', '25_30k', '30_35k', '35_40k', '40k_finish'
]
SEGMENT_END_KM = np.array(
    [5.0, 10.0, 15.0, 20.0, 21.0975, 25.0, 30.0, 35.0, 40.0, 42.195]
)
SEGMENT_MID_KM = np.array(
    [2.5, 7.5, 12.5, 17.5, 20.549, 23.049, 27.5, 32.5, 37.5, 41.098]
)

# ── City labels (adjust to match column values in your raw files) ─────────────
CITY_LABELS = ['Boston', 'Chicago', 'NewYork', 'Berlin']

print('Configuration OK')

## 2. Load raw data

In [ ]:
def load_marathon_results(raw_dir: Path, city_labels: list) -> pd.DataFrame:
    """
    Load and concatenate raw race result files from `raw_dir`.

    Expected file naming convention: {city}_{year}.csv
    Expected columns per file (adjust to actual source format):
        runner_id, finish_time_min, Sex, Age,
        split_5k, split_10k, split_15k, split_20k, split_half,
        split_25k, split_30k, split_35k, split_40k, split_finish
        (cumulative times in minutes)

    Returns
    -------
    pd.DataFrame with City and Year columns added.
    """
    frames = []
    for fpath in sorted(raw_dir.glob('*.csv')):
        # Parse city and year from filename
        stem = fpath.stem
        parts = stem.rsplit('_', 1)
        if len(parts) != 2:
            continue
        city, year = parts[0], parts[1]
        if city not in city_labels:
            continue
        df = pd.read_csv(fpath)
        df['City'] = city
        df['Year'] = int(year)
        frames.append(df)

    if not frames:
        raise FileNotFoundError(
            f'No valid result files found in {raw_dir}. '
            'Check the file naming convention.'
        )
    return pd.concat(frames, ignore_index=True)


raw_df = load_marathon_results(RAW_DIR, CITY_LABELS)
print(f'Raw records loaded: {len(raw_df):,}')
print(raw_df.head(2))

## 3. Reconstruct segment-wise paces

In [ ]:
# Cumulative split columns (adjust names to match your raw files)
SPLIT_COLS = [
    'split_5k', 'split_10k', 'split_15k', 'split_20k', 'split_half',
    'split_25k', 'split_30k', 'split_35k', 'split_40k', 'split_finish'
]
SPLIT_KM = SEGMENT_END_KM  # cumulative distances

# Segment distances (km)
SEG_DIST = np.diff(np.concatenate([[0.0], SPLIT_KM]))

# Segment cumulative times: prepend 0 (start) then diff
splits = raw_df[SPLIT_COLS].values  # shape (N, 10), cumulative minutes
cum_with_zero = np.hstack([np.zeros((len(raw_df), 1)), splits])
seg_times = np.diff(cum_with_zero, axis=1)  # segment elapsed time (min)

# Pace = time / distance (min/km)
pace_matrix = seg_times / SEG_DIST[np.newaxis, :]  # shape (N, 10)

# Store segment paces
for i, seg in enumerate(SEG_ORDER):
    raw_df[f'pace_{seg}'] = pace_matrix[:, i]

pace_cols = [f'pace_{s}' for s in SEG_ORDER]
print('Segment pace columns added:', pace_cols)

## 4. Quality filtering

In [ ]:
n_raw = len(raw_df)

# (a) Remove missing splits
mask_missing = raw_df[SPLIT_COLS].isnull().any(axis=1)

# (b) Remove physiologically implausible paces
mask_pace = (
    (raw_df[pace_cols] < PACE_MIN_KM).any(axis=1) |
    (raw_df[pace_cols] > PACE_MAX_KM).any(axis=1)
)

# (c) Remove outlier finish times
mask_finish = (
    (raw_df['finish_time_min'] < FINISH_MIN) |
    (raw_df['finish_time_min'] > FINISH_MAX)
)

mask_valid = ~(mask_missing | mask_pace | mask_finish)
df = raw_df[mask_valid].copy().reset_index(drop=True)

print(f'Raw:            {n_raw:>8,}')
print(f'Missing splits: {mask_missing.sum():>8,}')
print(f'Pace outliers:  {mask_pace.sum():>8,}')
print(f'Finish outliers:{mask_finish.sum():>8,}')
print(f'Retained:       {len(df):>8,}')
print()
print('By city and year:')
print(df.groupby(['City', 'Year']).size().to_string())

## 5. State variable definition

In [ ]:
# Early-race baseline: mean pace over first two segments (0–10 km)
early_cols = [f'pace_{SEG_ORDER[0]}', f'pace_{SEG_ORDER[1]}']
df['pace_early'] = df[early_cols].mean(axis=1)

# Full-race mean (alternative normalisation, Appendix B)
df['pace_mean_full'] = df[pace_cols].mean(axis=1)

# State variable y^B (primary, Paper §II.B): normalised by early pace
obsB_cols = []
for seg in SEG_ORDER:
    col = f'obsB_{seg}'
    df[col] = (df[f'pace_{seg}'] - df['pace_early']) / df['pace_early']
    obsB_cols.append(col)

# State variable y^A (alternative, Appendix B): normalised by full-race mean
obsA_cols = []
for seg in SEG_ORDER:
    col = f'obsA_{seg}'
    df[col] = (df[f'pace_{seg}'] - df['pace_mean_full']) / df['pace_mean_full']
    obsA_cols.append(col)

print('State variable columns added.')
print(f'  obsB (primary): {obsB_cols[:3]} ...')
print(f'  obsA (alt):     {obsA_cols[:3]} ...')
print()
print('obsB summary:')
print(df[obsB_cols].describe().round(3))

## 6. Performance group stratification

In [ ]:
# Quartile-based performance groups within each city
df['performance_group'] = df.groupby('City')['finish_time_min'].transform(
    lambda x: pd.qcut(x, q=4,
                      labels=['Q1_fastest', 'Q2', 'Q3', 'Q4_slowest'])
)
print('Performance group distribution:')
print(df['performance_group'].value_counts().sort_index())

## 7. Collapse (persistent wall) labeling

In [ ]:
def label_collapse(
    df: pd.DataFrame,
    state_cols: list,
    segment_end_km: np.ndarray,
    threshold: float = 0.10,
    late_start_idx: int = 5,
) -> pd.DataFrame:
    """
    Label persistent-wall collapse events (Paper §II.C).

    A collapse event is defined as the first occurrence of two consecutive
    segments with y > threshold for x >= 25 km.

    Parameters
    ----------
    df : DataFrame with state variable columns
    state_cols : ordered list of state variable column names
    segment_end_km : array of segment end distances (km)
    threshold : y_c value
    late_start_idx : first segment index to consider for collapse

    Returns
    -------
    DataFrame with added columns:
        wall_persistent (bool), wall_idx_persistent (int), wall_km_persistent (float)
    """
    n = len(df)
    n_seg = len(state_cols)
    y_mat = df[state_cols].values  # shape (N, n_seg)

    wall_flag = np.zeros(n, dtype=bool)
    wall_idx  = np.full(n, -1, dtype=int)
    wall_km   = np.full(n, np.nan)

    for k in range(late_start_idx, n_seg - 1):
        above_now  = y_mat[:, k]     > threshold
        above_next = y_mat[:, k + 1] > threshold
        new_event  = above_now & above_next & ~wall_flag
        wall_flag[new_event] = True
        wall_idx[new_event]  = k
        wall_km[new_event]   = segment_end_km[k]

    result = df.copy()
    result['wall_persistent']     = wall_flag
    result['wall_idx_persistent'] = wall_idx
    result['wall_km_persistent']  = wall_km
    return result


df = label_collapse(
    df,
    state_cols=obsB_cols,
    segment_end_km=SEGMENT_END_KM,
    threshold=Y_THRESHOLD,
    late_start_idx=LATE_START_IDX,
)

wall_rate = df['wall_persistent'].mean()
n_wall    = df['wall_persistent'].sum()
print(f'Persistent-wall rate: {wall_rate:.4f}  (N_wall = {n_wall:,})')

## 8. Export

In [ ]:
# ── Segment paces + state variables ───────────────────────────────────────────
meta_cols = [
    c for c in ['runner_id', 'City', 'Year', 'Sex', 'Age',
                'finish_time_min', 'performance_group',
                'pace_early', 'pace_mean_full']
    if c in df.columns
]
segment_df = df[meta_cols + pace_cols + obsB_cols + obsA_cols]
segment_file = OUTPUT_DIR / 'segment_paces_data.csv'
segment_df.to_csv(segment_file, index=False)
print(f'Saved: {segment_file}  shape={segment_df.shape}')

# ── Collapse labels ────────────────────────────────────────────────────────────
wall_cols = [
    c for c in ['runner_id', 'City', 'Year', 'Sex', 'Age',
                'finish_time_min', 'performance_group',
                'wall_persistent', 'wall_idx_persistent', 'wall_km_persistent']
    if c in df.columns
]
wall_df = df[wall_cols]
wall_file = OUTPUT_DIR / 'wall_labels_data.csv'
wall_df.to_csv(wall_file, index=False)
print(f'Saved: {wall_file}  shape={wall_df.shape}')

print('\n=== Dataset description (Paper §II) ===')
print(f'Total runners: {len(df):,}')
print(df.groupby(['City', 'Year']).size().to_string())
print(f'\nPersistent-wall rate (y_c={Y_THRESHOLD}): {wall_rate:.4f}')
print(f'N_wall = {n_wall:,}')